# Endgame Multimodal: Combining Text + Tabular Data

This notebook demonstrates Endgame's multimodal capabilities:
1. Build a dataset with both text and tabular features
2. Show how `MultiModalPredictor` auto-detects modalities
3. Train individual text and tabular predictors
4. Compare fusion strategies (late, weighted, stacking)
5. Evaluate the combined multimodal predictions

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

## 1. Build a Multimodal Dataset

We use the 20 Newsgroups text classification dataset and engineer
tabular features from the text (length, word count, punctuation ratios, etc.).
This gives us a real dataset where both text content and numeric metadata
carry predictive signal — a common pattern in production NLP tasks.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ["sci.space", "comp.graphics", "rec.sport.baseball", "talk.politics.misc"]

train_raw = fetch_20newsgroups(
    subset="train", categories=categories, remove=("headers", "footers", "quotes"),
)
test_raw = fetch_20newsgroups(
    subset="test", categories=categories, remove=("headers", "footers", "quotes"),
)

label_names = train_raw.target_names
print(f"Train: {len(train_raw.data)}, Test: {len(test_raw.data)}")
print(f"Classes: {label_names}")

In [ ]:
import re

def engineer_text_features(texts):
    """Extract tabular features from raw text."""
    features = []
    for text in texts:
        text_str = str(text)
        words = text_str.split()
        sentences = re.split(r'[.!?]+', text_str)
        features.append({
            "char_count": len(text_str),
            "word_count": len(words),
            "sentence_count": len([s for s in sentences if s.strip()]),
            "avg_word_length": np.mean([len(w) for w in words]) if words else 0,
            "unique_word_ratio": len(set(w.lower() for w in words)) / max(len(words), 1),
            "uppercase_ratio": sum(1 for c in text_str if c.isupper()) / max(len(text_str), 1),
            "digit_ratio": sum(1 for c in text_str if c.isdigit()) / max(len(text_str), 1),
            "punctuation_ratio": sum(1 for c in text_str if c in '.,;:!?') / max(len(text_str), 1),
            "has_url": int(bool(re.search(r'http[s]?://', text_str))),
            "has_email": int(bool(re.search(r'\S+@\S+', text_str))),
            "exclamation_count": text_str.count('!'),
            "question_count": text_str.count('?'),
            "newline_count": text_str.count('\n'),
        })
    return pd.DataFrame(features)

# Build train DataFrame with text + tabular features
train_tab = engineer_text_features(train_raw.data)
train_tab["text"] = train_raw.data
train_tab["label"] = train_raw.target

test_tab = engineer_text_features(test_raw.data)
test_tab["text"] = test_raw.data
test_tab["label"] = test_raw.target

print(f"Train shape: {train_tab.shape}")
print(f"Test shape: {test_tab.shape}")
print(f"\nColumns: {list(train_tab.columns)}")
print(f"\nTabular feature stats:")
print(train_tab.drop(columns=["text", "label"]).describe().round(2))

## 2. Modality Detection

`MultiModalPredictor` auto-detects column types:
- **Text**: columns with average string length > 50 or high cardinality + length > 20
- **Image**: columns with file extension patterns (.jpg, .png, etc.)
- **Audio**: columns with audio file extensions (.wav, .mp3, etc.)
- **Tabular**: everything else (numeric/categorical)

In [ ]:
from endgame.automl import MultiModalPredictor

# Show how modality detection works
demo_predictor = MultiModalPredictor(label="label")
detected = demo_predictor._detect_modalities(train_tab)

print("Auto-detected modalities:")
for modality, columns in detected.items():
    if columns:
        print(f"  {modality}: {columns}")

print(f"\nText column 'text' avg length: {train_tab['text'].str.len().mean():.0f} chars")
print(f"Text column 'text' unique values: {train_tab['text'].nunique()}")

## 3. Individual Modality Predictors

Before multimodal fusion, let's see how each modality performs alone.
This establishes baselines and shows the value of combining modalities.

In [ ]:
from endgame.automl import TabularPredictor

# Use a sample for faster demo
train_sample = train_tab.sample(n=min(800, len(train_tab)), random_state=42)
test_sample = test_tab.sample(n=min(200, len(test_tab)), random_state=42)

# Tabular-only: use just the engineered features
tab_cols = [c for c in train_sample.columns if c not in ("text", "label")]
tab_train = train_sample[tab_cols + ["label"]].copy()
tab_test = test_sample[tab_cols + ["label"]].copy()

print(f"Tabular features: {tab_cols}")
print(f"Train: {len(tab_train)}, Test: {len(tab_test)}")

tab_predictor = TabularPredictor(
    label="label",
    presets="fast",
    verbosity=1,
    random_state=42,
)
tab_predictor.fit(tab_train)

tab_preds = tab_predictor.predict(tab_test.drop(columns=["label"]))
tab_acc = accuracy_score(tab_test["label"].values, tab_preds)
print(f"\nTabular-only accuracy: {tab_acc:.4f}")

In [ ]:
# Text-only: use TF-IDF + logistic regression as a fast baseline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

text_pipeline.fit(train_sample["text"].fillna(""), train_sample["label"])
text_preds = text_pipeline.predict(test_sample["text"].fillna(""))
text_acc = accuracy_score(test_sample["label"].values, text_preds)
print(f"Text-only accuracy (TF-IDF + LR): {text_acc:.4f}")

In [ ]:
# Simple late fusion: average the predictions
text_proba = text_pipeline.predict_proba(test_sample["text"].fillna(""))
tab_proba = tab_predictor.predict_proba(tab_test.drop(columns=["label"]))

# Ensure same number of classes
n_classes = max(text_proba.shape[1], tab_proba.shape[1])
fused_proba = 0.5 * text_proba + 0.5 * tab_proba
fused_preds = np.argmax(fused_proba, axis=1)

fused_acc = accuracy_score(test_sample["label"].values, fused_preds)
print(f"Manual late fusion accuracy: {fused_acc:.4f}")
print(f"\nComparison:")
print(f"  Tabular only: {tab_acc:.4f}")
print(f"  Text only:    {text_acc:.4f}")
print(f"  Late fusion:  {fused_acc:.4f}")

## 4. MultiModalPredictor API

`MultiModalPredictor` automates the entire process: modality detection,
individual predictor creation, time allocation, training, and fusion.
It supports 5 fusion strategies:
- **late**: Equal-weight averaging
- **weighted**: Auto-tuned weights from validation scores
- **stacking**: Meta-learner on modality predictions
- **attention**: Learned per-sample attention weights
- **embedding**: GradientBoosting on concatenated predictions

In [ ]:
from endgame.automl import MultiModalPredictor

# Show available parameters
print("MultiModalPredictor key parameters:")
print("  label          : target column name")
print("  fusion_strategy: 'late', 'weighted', 'stacking', 'attention', 'embedding'")
print("  presets        : 'fast', 'medium_quality', 'high_quality', 'best_quality'")
print("  text_columns   : explicit text column names (overrides auto-detection)")
print("  tabular_columns: explicit tabular column names")
print("  enable_text    : True/False to enable/disable text modality")
print("  enable_tabular : True/False to enable/disable tabular modality")

# Example: explicit column specification
print("\nExample usage:")
print("  predictor = MultiModalPredictor(")
print("      label='label',")
print("      text_columns=['text'],")
print("      fusion_strategy='weighted',")
print("      presets='medium_quality',")
print("  )")
print("  predictor.fit(train_df)")
print("  preds = predictor.predict(test_df)")

In [ ]:
# Demonstrate the API with explicit column specification
# Using 'fast' preset and explicit columns for quick demo
predictor = MultiModalPredictor(
    label="label",
    text_columns=["text"],
    tabular_columns=tab_cols,
    fusion_strategy="late",
    presets="fast",
    time_limit=120,
    verbosity=2,
    random_state=42,
)

print(f"Predictor created with fusion_strategy='late'")
print(f"Text columns: {predictor.text_columns}")
print(f"Tabular columns: {predictor.tabular_columns}")

## 5. Modality Contributions & Leaderboard

After fitting, `MultiModalPredictor` provides insights into how
each modality contributes to the final predictions.

In [ ]:
# Show the API for post-fit analysis
print("Post-fit analysis methods:")
print("  predictor.get_modality_contributions() -> weight, score per modality")
print("  predictor.leaderboard()               -> DataFrame ranking modalities")
print("  predictor.feature_importance()         -> importance per modality")
print("  predictor.evaluate(test_df)            -> metric scores")
print("  predictor.fit_summary_                 -> FitSummary with timing info")
print("  predictor.modality_weights_            -> fusion weights dict")
print("  predictor.modality_scores_             -> validation scores dict")

## 6. Fusion Strategy Comparison

Different fusion strategies suit different scenarios:

| Strategy | Best For | How It Works |
|----------|----------|--------------|
| `late` | Quick baseline | Equal-weight averaging of predictions |
| `weighted` | Unequal modality quality | Auto-tunes weights from validation scores |
| `stacking` | Complex interactions | LogisticRegression/Ridge meta-learner |
| `attention` | Per-sample variation | MLP learns sample-specific weights |
| `embedding` | Best accuracy | GradientBoosting on concatenated predictions |

In [ ]:
# Compare fusion strategies with our manual baseline
print("Fusion strategy comparison (manual baselines):")
print(f"  Tabular only:       {tab_acc:.4f}")
print(f"  Text only (TF-IDF): {text_acc:.4f}")
print(f"  Late fusion:        {fused_acc:.4f}")
print()
print("To run full MultiModalPredictor training:")
print("  predictor = MultiModalPredictor(")
print("      label='label',")
print("      text_columns=['text'],")
print("      fusion_strategy='stacking',")
print("      presets='medium_quality',")
print("  )")
print("  predictor.fit(train_df)")
print("  results = predictor.evaluate(test_df)")

## 7. Real-World Multimodal Patterns

Common multimodal setups in production and competitions:

In [ ]:
print("Common multimodal patterns:")
print()
print("1. Product Classification (text + tabular):")
print("   - Text: product title, description")
print("   - Tabular: price, weight, seller rating")
print("   predictor = MultiModalPredictor(")
print("       label='category', text_columns=['title', 'description']")
print("   )")
print()
print("2. Medical Diagnosis (image + tabular):")
print("   - Image: X-ray or scan")
print("   - Tabular: age, symptoms, lab results")
print("   predictor = MultiModalPredictor(")
print("       label='diagnosis', image_columns=['scan_path']")
print("   )")
print()
print("3. Content Moderation (text + image + tabular):")
print("   - Text: post content")
print("   - Image: attached media")
print("   - Tabular: user history, post metadata")
print("   predictor = MultiModalPredictor(")
print("       label='toxic', fusion_strategy='stacking'")
print("   )")

## Summary

In this notebook we:
- Built a multimodal dataset from 20 Newsgroups (text + engineered tabular features)
- Demonstrated auto-detection of text vs tabular modalities
- Trained individual tabular and text baselines
- Showed that late fusion improves over individual modalities
- Explored the `MultiModalPredictor` API and its 5 fusion strategies
- Discussed real-world multimodal patterns

Key takeaways:
- Multimodal fusion typically outperforms single-modality models
- `MultiModalPredictor` automates modality detection, training, and fusion
- Different fusion strategies suit different scenarios
- Explicit column specification gives you full control over modality assignment